# EDA: Chest X-Ray Images (Pneumonia)

Run on Colab (no GPU needed). Downloads the data and makes `splits.csv` first.

In [ ]:
!git clone https://github.com/panidasrs/chest-xray-ai.git
%cd chest-xray-ai
!pip install -q kagglehub
!python prepare_data.py

## 1. How many images are in each split and class?

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

os.makedirs("results", exist_ok=True)
df = pd.read_csv("splits.csv")
df["class"] = df["label"].map({0: "normal", 1: "pneumonia"})

counts = df.groupby(["split", "class"]).size().unstack().loc[["train", "val", "test"]]
counts["pneumonia %"] = (100 * counts["pneumonia"] / counts.sum(axis=1)).round(1)
counts

In [ ]:
counts[["normal", "pneumonia"]].plot.bar(rot=0, figsize=(6, 3.5), color=["#4c9be8", "#e8794c"])
plt.title("Images per split")
plt.ylabel("images")
plt.tight_layout()
plt.savefig("results/eda_class_balance.png", dpi=120)
plt.show()

## 2. Pneumonia type (from the file name)

Pneumonia file names contain `bacteria` or `virus`. The model only predicts normal vs pneumonia, but it is useful to know the mix.

In [ ]:
pneu = df[df["label"] == 1].copy()
pneu["type"] = np.where(pneu["path"].str.contains("virus"), "virus", "bacteria")
pneu.groupby(["split", "type"]).size().unstack()

## 3. How many images per patient?

This is why the train/val split is done by patient: some children have several X-rays.

In [ ]:
per_patient = df[df["split"] != "test"].groupby(["class", "patient"]).size()
print(per_patient.groupby("class").describe()[["count", "mean", "max"]])

per_patient.groupby("class").value_counts().unstack(0).head(6)

## 4. Image size and color mode

Images are resized to 224x224 for the model, so it is worth checking how big and how different they are.

In [ ]:
rows = []
for path in df["path"]:
    with Image.open(path) as img:  # opening is fast, pixels are not loaded
        rows.append({"width": img.size[0], "height": img.size[1], "mode": img.mode})
sizes = pd.concat([df[["split", "class"]].reset_index(drop=True), pd.DataFrame(rows)], axis=1)
sizes["aspect"] = (sizes["width"] / sizes["height"]).round(2)

print(sizes["mode"].value_counts(), "\n")
sizes.groupby("class")[["width", "height", "aspect"]].describe().T.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for name, g in sizes.groupby("class"):
    ax.scatter(g["width"], g["height"], s=4, alpha=0.4, label=name)
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
ax.set_title("Original image sizes")
ax.legend()
plt.tight_layout()
plt.savefig("results/eda_image_sizes.png", dpi=120)
plt.show()

## 5. Example images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for row, (name, g) in enumerate(df[df["split"] == "train"].groupby("class")):
    for col, path in enumerate(g.sample(5, random_state=1)["path"]):
        axes[row, col].imshow(Image.open(path).convert("L"), cmap="gray")
        axes[row, col].set_title(name)
        axes[row, col].axis("off")
plt.tight_layout()
plt.savefig("results/eda_examples.png", dpi=100)
plt.show()

## 6. Brightness per class

If the two classes have very different brightness, the model could learn brightness instead of the disease (a shortcut).

In [ ]:
def mean_brightness(path):
    with Image.open(path) as img:
        return np.asarray(img.convert("L").resize((224, 224))).mean()

sample = df[df["split"] == "train"].groupby("class").sample(300, random_state=0)
sample["brightness"] = sample["path"].apply(mean_brightness)

print(sample.groupby("class")["brightness"].describe().round(1))
sample.groupby("class")["brightness"].plot.hist(bins=30, alpha=0.5, legend=True, figsize=(6, 3.5))
plt.xlabel("mean pixel value (0-255)")
plt.title("Brightness of 300 train images per class")
plt.tight_layout()
plt.savefig("results/eda_brightness.png", dpi=120)
plt.show()

## 7. What I learned

_(fill in after running)_

- Class balance: 
- Images per patient: 
- Image sizes: 
- Brightness: 